# 🌡️ PyTorch RNN · LSTM · GRU — 1시간 뒤 기온 맞히기

**건축공학딥러닝프로그래밍 · PyTorch RNN·LSTM·GRU (기온 예측) · 실습 노트북 (실습본 · 빈칸 채우기)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baekja/Lecture_DL-AE/blob/main/PyTorch_RNN_Jena/jena_rnn_practice.ipynb) ← **실습본**(빈칸 채우기) &nbsp;&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baekja/Lecture_DL-AE/blob/main/PyTorch_RNN_Jena/jena_rnn_complete.ipynb) ← **완성본**(정답·실행결과 포함)

> 지금까지는 그림 한 장을 보고 답했습니다. 오늘의 데이터는 **순서가 있는 숫자**, 독일 예나(Jena)의 기상관측소가 8년 동안 10분마다 잰 기온입니다.
> 지난 24시간의 기온을 보고 **1시간 뒤 기온**을 맞히는 RNN · LSTM · GRU 를 만들고, "지금 기온 그대로"라는 단순 예측보다 몇 °C 나은지 확인합니다.
> 그 전에 이론 시간의 RNN 식과 LSTM 게이트 식을 손으로 계산해, `nn.RNN` · `nn.LSTM` 과 **똑같은 숫자**가 나오는지부터 봅시다.

## 📌 강의 중점
- 시계열을 **창(window)** 으로 잘라 "24시간 → 1시간 뒤" 학습 데이터 만들기 · 시간 순서대로 나누기 · 정규화
- 이론 시간의 RNN · LSTM 식을 손으로 계산해 `nn.RNN` · `nn.LSTM` 결과와 비교 · 파라미터 수가 RNN 의 4배(LSTM) · 3배(GRU)인 이유
- 강의노트의 RNN · LSTM · GRU 모델 클래스로 기온 예측 · **단순 예측(기준점)** 과 비교해 실력 재기

## 🎯 학습 목표
1. 시계열을 `[샘플 수, 시점 수, 변수 수]` 모양의 입력 창으로 만들고, 시간 순서대로 학습 · 검증 · 시험 데이터로 나눌 수 있다.
2. RNN 의 은닉 상태 식 $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b)$ 를 코드로 계산해 `nn.RNN` 결과와 같음을 확인할 수 있다.
3. LSTM · GRU 의 게이트 수로 파라미터 수를 계산하고 코드로 확인할 수 있다.
4. 세 모델을 학습시켜 시험 데이터의 평균 오차(°C)를 구하고, 단순 예측(기준점)과 비교해 해석할 수 있다.

## 📋 이 노트북 사용법

| 표시 | 뜻 |
|:-:|---|
| ▶ | 셀 실행 — 셀을 클릭하고 `Shift + Enter` |
| 🎬 EP*n* | 이 부분을 설명하는 강의 영상 에피소드 (e-campus) |
| ⏸ | 영상을 멈추고 직접 실행해 보는 지점 |
| ✅ / ❌ | 자가검증 결과 — ❌ 가 나오면 힌트를 읽고 고쳐 보세요 |
| 🔸 | 실험 결과 안내 (오류가 아님) |
| ✏️ | 빈칸 `____` 을 채워야 하는 줄 (주석 끝의 힌트 참고) |

**주석(`#` 뒤의 글) 읽는 법** — 실행에는 영향이 없는 설명입니다.
- `①②③`, `1️⃣2️⃣3️⃣4️⃣` 순서(단계) · `💡` 이론 연결(RNN · LSTM · GRU 강의) · `📐` 데이터 모양(shape) · `⚠️` 흔한 실수 · `✅` 기대 결과 · `⏱` 걸린 시간

**Colab에서 처음 열었다면**
1. 메뉴 **파일 → Drive에 사본 저장** 을 먼저 누르세요. (사본에서 작업해야 내 Drive에 저장됩니다)
2. 4장과 5장에 학습이 있습니다. CPU 그대로 두어도 모든 셀이 실행되고, **GPU 런타임**(메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**)이면 더 빠릅니다.
3. 위에서부터 **차례대로** 실행하세요. 런타임이 초기화되어 변수가 사라지면 처음부터 다시 실행하면 됩니다.
4. ✏️ 표시된 줄의 `____` 을 채운 뒤 실행하세요. **'모두 실행'은 첫 빈칸에서 멈춥니다** — 하나씩 채우며 내려가세요.

**🎬 영상과 함께 보는 법 (반복 학습 추천 순서)**
1회차 영상 전체 보기 → 2회차 실습본을 열고 ⏸ 마다 멈추며 따라 치기 → 3회차 영상 없이 빈칸 채우기 → 모든 ✅ 통과 → 5장 도전 과제

---
## 0. 준비 — 라이브러리 불러오기  🎬 EP1

이 셀에서 할 일: PyTorch와 pandas(표 데이터 도구)를 불러오고, 계산할 장치(CPU 또는 GPU)를 정합니다.

> 💡 **새 파이썬 문법 — pandas**
>
> pandas 는 엑셀 표 같은 데이터를 다루는 도구입니다. `import pandas as pd` 로 불러와 `pd.read_csv(...)` 처럼 씁니다.
> `from torch.utils.data import TensorDataset` 은 '텐서 묶음 → 데이터셋' 도구입니다 (4장에서 씁니다).

In [ ]:
import torch                                   # PyTorch: 딥러닝 도구 상자
from torch import nn                           # nn = 신경망 부품(층) 모음
from torch.utils.data import TensorDataset, DataLoader  # 텐서 → 데이터셋 → 미니배치
import pandas as pd                            # 표(CSV) 데이터 도구 (별명 pd)
import matplotlib.pyplot as plt                # 그래프 도구 (별명 plt)
import time                                    # 걸린 시간 재기 (⏱)

torch.set_printoptions(precision=4, sci_mode=False)      # 소수 4자리, 지수 표기 끄기
device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU가 있으면 GPU, 없으면 CPU
print("PyTorch 버전:", torch.__version__)                # Colab에는 이미 설치됨
print("계산 장치:", device)                              # cuda = GPU, cpu = CPU

### 자가검증 도우미 `check()` · `check_range()` 준비
아래 셀은 결과가 맞는지 ✅ / ❌ 로 알려 주는 **도우미 함수**입니다. 지난 실습과 같습니다.
- `check()` : 기대값과 허용오차(`tol`) 이내로 같으면 ✅ (`tol=0` 이면 완전히 같아야 ✅)
- `check_range()` : 오차처럼 **범위 안에 들어오면** ✅ (컴퓨터 · GPU 마다 조금씩 달라지는 값에 씁니다)

지금은 내용을 몰라도 괜찮습니다 — **실행만** 해 두세요.

In [ ]:
def check(name, value, expected, tol=0.01):  # 자가검증 (이름, 결과, 기대값, 허용오차)
    """value 가 expected 와 tol 이내로 같으면 ✅, 다르면 ❌ 를 보여 줍니다."""
    value = torch.as_tensor(value, dtype=torch.float32)        # 숫자·리스트 → 텐서
    expected = torch.as_tensor(expected, dtype=torch.float32)  # 기대값도 텐서로
    if value.shape != expected.shape:                          # ⚠️ 모양부터 다르면
        print(f"❌ {name}: 모양 {tuple(value.shape)} ≠ 기대 {tuple(expected.shape)}")
        return                                                 # 여기서 끝
    gap = (value - expected).abs().max().item()                # 가장 큰 차이
    mark = "✅" if gap <= tol else "❌"                         # 허용오차 안이면 통과
    tol_s = f"{tol:.6f}".rstrip("0").rstrip(".")               # 0.000001 처럼 보기 좋게
    print(f"{mark} {name}: 최대 차이 {gap:.6f} (허용 {tol_s})")  # 결과 출력

def check_range(name, value, lo, hi):        # 자가검증 (이름, 결과, 최솟값, 최댓값)
    """value 가 lo ~ hi 범위 안이면 ✅, 벗어나면 ❌ 를 보여 줍니다."""
    value = float(value)                                       # 텐서·숫자 → 파이썬 실수
    mark = "✅" if lo <= value <= hi else "❌"                  # 범위 안이면 통과
    print(f"{mark} {name}: {value:.4f} (기대 범위 {lo} ~ {hi})")  # 결과 출력

---
## 1. 데이터 — 예나(Jena) 기상 관측 기록

이 셀에서 할 일: 독일 예나의 막스 플랑크 생물지구화학 연구소 기상관측소가 **2009년 1월부터 2016년 12월까지 10분마다** 기록한 데이터(CSV)를 인터넷에서 바로 읽습니다.
- TensorFlow 공식 튜토리얼 「시계열 예측」과 『케라스 창시자에게 배우는 딥러닝』에서도 시계열 예제로 쓰는 데이터입니다.
- 기온 · 기압 · 습도 · 풍속 등 측정값이 14가지 있습니다. 오늘은 주로 **기온 `T (degC)`** 하나를 씁니다.

> 💡 **새 파이썬 문법 — 표(DataFrame)**
>
> `pd.read_csv(주소)` 는 CSV(쉼표로 나눈 표) 파일을 읽어 **표(DataFrame)** 로 만듭니다.
> `df.shape` 는 (행 수, 열 수), `df.head(3)` 은 앞 3줄, `df["열 이름"]` 은 열 하나, `df[["열1", "열2"]]` 는 열 여러 개입니다.

In [ ]:
URL = ("https://storage.googleapis.com/tensorflow/"           # TensorFlow 공개 데이터
       "tf-keras-datasets/jena_climate_2009_2016.csv.zip")    # zip 안에 CSV 파일 1개
df = pd.________(URL)                       # ✏️ 내려받아 표로 읽기 (13.5 MB, 몇 초) (힌트: CSV 읽기)
print("표 모양 (행, 열):", df.shape)        # 📐 10분 간격 기록 × (날짜 + 측정값 14개)
df[["Date Time", "T (degC)", "p (mbar)", "rh (%)"]].head(3)  # 앞 3줄: 날짜, 기온, 기압, 습도

🔍 **결과 읽는 법** — 42만 줄 ≈ 8년 × 365일 × 하루 144번(10분 간격)입니다. `T (degC)` 는 섭씨 기온, `p (mbar)` 는 기압(밀리바), `rh (%)` 는 상대 습도입니다.

### 1-1. 1시간 간격으로 줄이고, 그래프로 보기 ⏸
10분 간격은 너무 촘촘합니다. TensorFlow 튜토리얼처럼 **6줄마다 1줄**(정각 기록)만 남겨 1시간 간격으로 만듭니다.

> 💡 **새 파이썬 문법 — 건너뛰는 슬라이싱 `[시작::간격]`**
>
> `df[5::6]` 은 5번 줄부터 끝까지 **6줄마다 하나씩** 고릅니다. 0번 줄이 00:10 이므로 5번 줄이 01:00 정각입니다.

In [ ]:
df = df[____]                               # ✏️ 6줄마다 1줄 = 매시 정각 (힌트: 시작::간격)
FEATURES = ["T (degC)"]                     # 쓸 측정값: 기온 하나 (5장에서 바꿔 봅니다)
data = torch.tensor(df[FEATURES].values, dtype=torch.float32)  # 표 → 텐서  📐 [시간 수, 1]
print("1시간 간격:", len(df), "줄 · 텐서 모양:", tuple(data.shape))  # 약 7만 시간
print("처음:", df["Date Time"].iloc[0], "· 끝:", df["Date Time"].iloc[-1])  # 일.월.년 시각
print(f"기온: 최저 {data.min():.1f} °C · 최고 {data.max():.1f} °C")  # 기온 범위

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.4))   # 그래프 2개를 가로로
ax1.plot(data[:, 0], lw=0.3)                # 8년 전체: 여름엔 높고 겨울엔 낮은 1년 주기
ax2.plot(data[4320:4560, 0], ".-", ms=3)    # 여름 10일(240시간): 낮과 밤의 하루 주기
ax1.set_title("8 years, hourly"); ax2.set_title("10 days in summer")       # 제목 (영문)
ax1.set_xlabel("hour"); ax2.set_xlabel("hour"); ax1.set_ylabel("T (°C)")  # 축 이름
plt.tight_layout()                          # 간격 정리
plt.show()                                  # 그리기

🔍 **결과 읽는 법** — 왼쪽은 1년 주기(여름 · 겨울), 오른쪽은 하루 주기(낮 · 밤)가 보입니다. 그리고 **바로 앞 시간과 다음 시간의 기온은 거의 비슷**합니다. 이 성질이 4장의 '기준점'을 만만치 않게 만듭니다.
(끝 시각이 정각이 아닌 23:10 인 것은 기록 중간에 빠진 줄이 조금 있어서입니다. TensorFlow 튜토리얼도 같은 방법을 쓰며, 거의 1시간 간격이면 충분합니다.)

---
## 2. 창(window) 만들기 — "24시간을 보고 1시간 뒤를 맞힌다"

이 셀에서 할 일: 긴 기온 기록을 **입력 창 24시간 + 정답 1개** 묶음으로 잘라 학습 샘플을 만듭니다. 창을 한 시간씩 밀 때마다 샘플이 하나씩 생깁니다.
- 이론 시간의 **many-to-one** 구조입니다: 여러 시점의 입력 $x_1, \dots, x_{24}$ → 예측 하나 $\hat y$
- `ahead` 는 창이 끝난 뒤 몇 시간째를 맞힐지입니다. `ahead=1` 이면 바로 다음 시간입니다.

| 샘플 | 입력 창 (24시간) | 정답 (`ahead=1`) |
|:-:|:-:|:-:|
| 0 | 0 ~ 23번 시간 | 24번 시간 |
| 1 | 1 ~ 24번 시간 | 25번 시간 |
| … | 한 시간씩 밀기 | … |

In [ ]:
def make_windows(series, width=24, ahead=1):   # 시계열 → (입력 창들, 정답들)
    X, y = [], []                              # 입력 창, 정답을 모을 빈 리스트
    for i in range(len(series) - width - ahead + 1):  # 창의 시작 위치 i 를 한 칸씩 밀며
        end = i + width                        # 창이 끝나는 위치 (이 칸은 창에 안 들어감)
        X.append(series[i:end])                # 입력: i ~ end-1 (24시간)  📐 [24, 변수 수]
        y.append(series[end + _________, :1])  # ✏️ 정답: 기온 (0번 열) (힌트: ahead=1 이면 0)
    return torch.stack(X), torch.stack(y)      # 리스트 → 텐서  📐 [N, 24, 변수 수], [N, 1]

### 2-1. 숫자 0 ~ 9 로 손 확인 ⏸
기온 대신 0, 1, …, 9 를 넣어 보면 창이 눈에 보입니다. `width=3` 이면 [0, 1, 2] → 3, [1, 2, 3] → 4, … 이렇게 **7개**가 나와야 합니다 (10 − 3 − 1 + 1 = 7).

In [ ]:
toy = torch.arange(10.0).view(-1, 1)        # 0, 1, …, 9 (시점 10개 · 변수 1개)  📐 [10, 1]
Xs, ys = make_windows(toy, width=3, ahead=1)  # 창 3칸 → 바로 다음 칸
print("창 개수:", len(Xs))                    # 10 - 3 - 1 + 1 = 7
print("첫 창:", Xs[0, :, 0].tolist(), "→ 정답", ys[0].tolist())   # [0, 1, 2] → [3]
print("끝 창:", Xs[-1, :, 0].tolist(), "→ 정답", ys[-1].tolist())  # [6, 7, 8] → [9]
check("창 개수", len(Xs), 7, tol=0)          # ✅ 7개
check("첫 창의 정답", ys[0], [3.0], tol=0)  # ✅ 3
Xs2, ys2 = make_windows(toy, width=3, ahead=2)  # 정답을 한 칸 더 뒤로
check("ahead=2 이면 첫 정답은 4", ys2[0], [4.0], tol=0)  # ✅ 4

---
## 3. 시간 순서대로 나누기 · 정규화

이 셀에서 할 일: 앞 70 % 를 **학습**, 다음 20 % 를 **검증**, 마지막 10 % 를 **시험** 데이터로 나눕니다 (TensorFlow 튜토리얼과 같은 비율).
- ⚠️ **무작위로 섞어 나누면 안 됩니다.** 시험 기간 바로 앞뒤의 시간이 학습에 섞여, 미래를 보고 맞히는 셈이 됩니다. **과거로 배우고 미래로 시험**합니다.
- **정규화** = (값 − 평균) ÷ 표준편차. 평균과 표준편차는 **학습 구간에서만** 구합니다 (시험 구간을 미리 보지 않도록).

In [ ]:
n = len(data)                               # 전체 시간 수 (약 7만)
n_train, n_val = int(n * 0.7), int(n * 0.9) # 경계: 70 % 지점, 90 % 지점
mean = data[:n_train].mean(dim=0)           # 학습 구간의 평균 (변수마다)
std = data[:n_train].std(dim=0)             # 학습 구간의 표준편차 (변수마다)
z = ___________________                     # ✏️ 정규화 (힌트: (값 − 평균) ÷ 표준편차)
z_train, z_val, z_test = z[:n_train], z[n_train:n_val], z[n_val:]  # 시간 순서대로 3토막
print(f"학습 / 검증 / 시험: {len(z_train)} / {len(z_val)} / {len(z_test)} 시간")  # 7 : 2 : 1
print(f"학습 구간 기온: 평균 {mean[0]:.2f} °C · 표준편차 {std[0]:.2f} °C")  # 정규화에 쓴 값

### 3-1. 세 구간을 각각 창으로 자르기
창은 **구간 안에서만** 만듭니다. 그래야 학습 창이 시험 기간에 걸치지 않습니다.

In [ ]:
X_train, y_train = make_windows(z_train)    # 학습용 창 (24시간 → 1시간 뒤)
X_val, y_val = make_windows(z_val)          # 검증용 창
X_test, y_test = make_windows(z_test)       # 시험용 창
print("학습:", tuple(X_train.shape), "→", tuple(y_train.shape))  # 📐 [N, 24, 1] → [N, 1]
print("검증:", tuple(X_val.shape), "· 시험:", tuple(X_test.shape))  # 📐 [N, 24, 1]

🔍 **결과 읽는 법** — 입력 모양 `[샘플 수, 24, 1]` 은 **[배치, 시점, 변수]** 순서입니다. 다음 장의 `nn.RNN(..., batch_first=True)` 가 바로 이 순서를 받습니다.

---
## 4. 기준점 — "지금 기온 그대로" 예측

모델을 만들기 전에, 아주 단순한 예측의 점수부터 잽니다: **1시간 뒤 기온 = 지금(창의 마지막 시간) 기온**.
- 『케라스 창시자에게 배우는 딥러닝』은 이것을 **'상식 수준의 기준점'** 이라 부릅니다. 딥러닝 모델은 **이 점수를 이겨야** 쓸모가 있습니다.
- 점수는 **평균 절대 오차(MAE)** = |예측 − 정답| 의 평균입니다. 정규화된 값의 차이에 표준편차를 곱하면 °C 로 돌아옵니다.

In [ ]:
def persistence_mae(X, y):                  # 기준점: '지금 기온 그대로'의 평균 절대 오차 (°C)
    pred = X[:, ___, :1]                     # ✏️ 창의 마지막 시간의 기온 = 예측 (힌트: 마지막 시점)
    return ((pred - y).abs().mean() * std[0]).item()  # |예측 − 정답| 평균 × 표준편차 → °C

base_val = persistence_mae(X_val, y_val)    # 검증 구간 점수
base_test = persistence_mae(X_test, y_test) # 시험 구간 점수
print(f"기준점 평균 오차: 검증 {base_val:.2f} °C · 시험 {base_test:.2f} °C")  # 이길 목표
check_range("기준점 시험 오차 (°C)", base_test, 0.6, 0.9)  # ✅ 1 °C 미만

🔍 **결과 읽는 법** — "1시간 뒤도 지금과 같다"고만 해도 평균 0.74 °C 밖에 틀리지 않습니다. 꽤 좋은 점수라 이기기 쉽지 않습니다. 4장에서 RNN · LSTM · GRU 가 이 오차를 얼마나 줄이는지 봅시다.

---
## 5. RNN 한 칸을 손으로 — `nn.RNN` 과 같은 숫자가 나올까?  🎬 EP2

이론 시간의 RNN 식입니다 (편향 $b$ 포함):

$$h_t = \tanh(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b), \qquad h_0 = 0$$

- 같은 $W_{xh}, W_{hh}, b$ 를 **모든 시점이 함께 씁니다** (파라미터 공유).
- 이 셀에서 할 일: 입력 1개 · 은닉 상태 2칸짜리 **아주 작은 RNN** 을 만들고 가중치를 들여다봅니다.
- PyTorch 의 이름: $W_{xh}$ = `weight_ih_l0`, $W_{hh}$ = `weight_hh_l0`, 편향은 `bias_ih_l0` 와 `bias_hh_l0` 두 개로 나뉘어 있고 **더하면 식의 $b$** 입니다.

In [ ]:
torch.manual_seed(0)                        # 같은 초기 가중치 (재현용)
rnn = nn.RNN(1, 2, ___________=True)        # ✏️ 입력 1개 → 은닉 상태 2칸 (힌트: 배치가 먼저)
W_xh = rnn.weight_ih_l0.detach()            # 입력 → 은닉 가중치  📐 [2, 1]
W_hh = rnn.weight_hh_l0.detach()            # 은닉 → 은닉 가중치  📐 [2, 2]
b = (rnn.bias_ih_l0 + rnn.bias_hh_l0).detach()  # 편향 두 개를 더한 식의 b  📐 [2]
print("W_xh =", W_xh.view(-1))              # 숫자 2개
print("W_hh =", W_hh)                       # 2 × 2 = 숫자 4개
print("b    =", b)                          # 숫자 2개

### 5-1. 기온 3시간을 넣어 보기 — `out` 과 `hn`
첫 학습 창의 처음 3시간(정규화된 기온)을 넣습니다. 입력 모양은 [배치 1, 시점 3, 변수 1] 입니다.
- `out` : **모든 시점**의 은닉 상태 `[1, 3, 2]`
- `hn` : **마지막** 은닉 상태 `[층 1, 배치 1, 2]`

In [ ]:
x3 = X_train[:1, :3, :]                     # 첫 창의 처음 3시간  📐 [1, 3, 1]
print("입력 x1, x2, x3 =", x3.view(-1))      # 정규화된 기온 3개
print("°C 로 되돌리면  =", x3.view(-1) * std[0] + mean[0])  # 새해 첫날 새벽 기온
with torch.no_grad():                       # 계산만 (미분 추적 끄기)
    out, hn = rnn(x3)                       # RNN 에 3시간을 차례로 넣기
print("out 모양:", tuple(out.shape), "· hn 모양:", tuple(hn.shape))  # 📐 (1, 3, 2) · (1, 1, 2)
print("out =", out[0])                      # 시점 1, 2, 3 의 은닉 상태 (각 2칸)

### 5-2. 손 계산 — 식을 3번 적용 ⏸
`for` 문으로 시점 1, 2, 3 에 이론 식을 그대로 적용합니다. `@` 는 행렬 곱입니다.

In [ ]:
h = torch.zeros(2)                          # h0 = 0 (처음엔 기억이 없음)
for t in range(3):                          # 시점 1, 2, 3 차례로
    x_t = x3[0, t]                          # 이번 시점의 입력 (기온 1개)  📐 [1]
    h = torch.____(W_xh @ x_t + W_hh @ h + b)  # ✏️ 💡 이론 시간의 식 그대로 (힌트: -1~1 활성화)
    print(f"h{t + 1} =", h)                 # 손 계산 결과 (은닉 상태 2칸)
check("손 계산 h3 = nn.RNN 의 hn", h, hn[0, 0], tol=1e-6)  # ✅ 같아야 함
check("out 의 마지막 줄 = hn", out[0, -1], hn[0, 0], tol=0)  # ✅ 완전히 같음

🔍 **결과 읽는 법** — 손으로 구한 $h_1, h_2, h_3$ 가 `out` 의 세 줄과 같고, $h_3$ 는 `hn` 과 같습니다. 세 번 모두 **같은 가중치**를 썼죠 (파라미터 공유). `nn.RNN` 은 이 `for` 문을 안에서 대신 돌려 줄 뿐입니다. 은닉 상태는 tanh 때문에 늘 −1 ~ 1 사이입니다.

---
## 6. 강의노트의 `RNNModel` — 마지막 시점의 은닉 상태 → 기온 1개

강의노트의 클래스를 거의 그대로 씁니다. 바꾼 곳은 두 군데입니다.

| | 강의노트 (손글씨 숫자 분류) | 이 노트북 (기온 예측) |
|---|---|---|
| `input_size` | 28 (그림의 한 행) | **1** (기온) |
| 마지막 출력 | `num_classes` = 10 (점수 10개) | **`output_size` = 1** (기온 1개) |

- `out[:, -1, :]` : 24시간 중 **마지막 시점**의 은닉 상태만 꺼내 완전연결층(`fc`)에 넣습니다 = many-to-one

> 💡 **다시 보는 파이썬 문법 — `class` · `super().__init__()`**
>
> 강의노트의 `super(RNNModel, self).__init__()` 와 `super().__init__()` 는 같은 뜻입니다 (짧게 쓴 것).

In [ ]:
class RNNModel(nn.Module):                  # 강의노트의 RNN 모델 (분류 → 기온 예측)
    def __init__(self, input_size, hidden_size, num_layers, output_size):  # 크기 4가지
        super().__init__()                  # nn.Module 준비
        self.hidden_size = hidden_size      # 은닉 상태 칸 수
        self.num_layers = num_layers        # 쌓을 RNN 층 수
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)  # 순환층
        self.fc = nn.Linear(hidden_size, output_size)   # 은닉 상태 → 기온 1개

    def forward(self, x):                   # x 📐 [배치, 24, 변수 수]
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # h0 = 0
        out, hn = self.rnn(x, h0)           # 📐 out [배치, 24, 은닉]
        out = out[:, ___, :]                 # ✏️ 마지막 시점만  📐 [배치, 은닉] (힌트: 마지막 시점)
        return self.fc(out)                 # 📐 [배치, 1] = 1시간 뒤 기온 (정규화된 값)

In [ ]:
torch.manual_seed(0)                        # 재현용
model = RNNModel(1, 32, 1, 1)               # 입력 1 · 은닉 32 · 1층 · 출력 1
pred = model(X_train[:5])                   # 창 5개를 한꺼번에  📐 [5, 24, 1]
print("출력 모양:", tuple(pred.shape))       # 📐 (5, 1): 창마다 기온 1개
print("파라미터 수:", sum(p.numel() for p in model.parameters()))  # 학습할 숫자 개수
check("출력 모양", list(pred.shape), [5, 1], tol=0)  # ✅

🔍 **결과 읽는 법** — 아직 학습 전이라 예측값은 엉터리입니다. 모양만 맞으면 됩니다: 창 5개 → 기온 5개. 파라미터 수가 왜 이 숫자인지는 다음 장에서 셉니다.

---
## 7. LSTM — 셀 상태와 게이트 3개  🎬 EP3

RNN 은 은닉 상태 하나에 모든 기억을 담아, 오래된 정보일수록 흐려집니다 (기울기 소실). LSTM 은 **셀 상태 $c_t$** 라는 장기 기억 통로를 두고, 게이트 3개로 무엇을 지우고 · 더하고 · 내보낼지 정합니다.

| 이름 | 식 (이론 시간) | 하는 일 |
|---|---|---|
| 망각 게이트 $f_t$ | $\sigma(W_{xh_f}x_t + W_{hh_f}h_{t-1} + b_f)$ | 지난 셀 상태를 얼마나 **남길지** (0 ~ 1) |
| 입력 게이트 $i_t$ | $\sigma(W_{xh_i}x_t + W_{hh_i}h_{t-1} + b_i)$ | 새 정보를 얼마나 **더할지** (0 ~ 1) |
| 임시 셀 상태 $\tilde c_t$ | $\tanh(W_{xh_g}x_t + W_{hh_g}h_{t-1} + b_g)$ | 더할 새 정보 (−1 ~ 1) |
| 출력 게이트 $o_t$ | $\sigma(W_{xh_o}x_t + W_{hh_o}h_{t-1} + b_o)$ | 셀 상태를 얼마나 **내보낼지** (0 ~ 1) |
| 셀 상태 | $c_t = f_t \otimes c_{t-1} + i_t \otimes \tilde c_t$ | 장기 기억 |
| 은닉 상태 | $h_t = o_t \otimes \tanh(c_t)$ | 단기 기억 = 출력 |

PyTorch 는 네 묶음(입력 · 망각 · 임시 셀 · 출력 순서)의 가중치를 **세로로 이어 붙여** 한 행렬에 담습니다. 그래서 가중치 행렬이 RNN 의 4배 크기입니다.

In [ ]:
torch.manual_seed(0)                        # 재현용
lstm = nn.LSTM(1, 2, batch_first=True)      # 입력 1개 → 은닉 상태 2칸
W = lstm.weight_ih_l0.detach()              # 입력 쪽 가중치 4묶음  📐 [8, 1] = 4 × 2
U = lstm.weight_hh_l0.detach()              # 은닉 쪽 가중치 4묶음  📐 [8, 2]
b = (lstm.bias_ih_l0 + lstm.bias_hh_l0).detach()  # 편향 4묶음  📐 [8]
print("W 모양:", tuple(W.shape), "· U 모양:", tuple(U.shape), "· b 모양:", tuple(b.shape))
with torch.no_grad():                       # 계산만
    out, (hn, cn) = lstm(x3)                # 같은 3시간 · (은닉 상태, 셀 상태)를 묶어 돌려줌
print("hn =", hn[0, 0], "· cn =", cn[0, 0])  # 마지막 은닉 상태 · 셀 상태

### 7-1. LSTM 손 계산 ⏸
> 💡 **새 파이썬 문법 — `.chunk(4)` 와 한 번에 담기**
>
> `텐서.chunk(4)` 는 텐서를 4토막으로 나눕니다. `i, f, g, o = …` 처럼 쓰면 네 토막이 변수 4개에 차례로 담깁니다.

In [ ]:
h, c = torch.zeros(2), torch.zeros(2)       # h0 = 0, c0 = 0
for t in range(3):                          # 시점 1, 2, 3
    i, f, g, o = (W @ x3[0, t] + U @ h + b).chunk(4)  # 곱하고 더한 8개 → 2개씩 4토막
    i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)  # 게이트 3개: 0 ~ 1
    g = torch.tanh(g)                       # 임시 셀 상태: -1 ~ 1
    c = _____ + i * g                       # ✏️ 셀 상태 (힌트: 망각 게이트 × 지난 셀)
    h = o * _____________                   # ✏️ 은닉 상태 (힌트: 셀 상태를 -1~1 로)
check("손 계산 h3 = nn.LSTM 의 hn", h, hn[0, 0], tol=1e-6)  # ✅
check("손 계산 c3 = nn.LSTM 의 cn", c, cn[0, 0], tol=1e-6)  # ✅

🔍 **결과 읽는 법** — LSTM 도 결국 '곱하고 더한 뒤 시그모이드 · tanh' 입니다. 셀 상태 식이 **더하기**라서, 망각 게이트가 1 에 가까우면 기억이 거의 그대로 다음 시점으로 넘어갑니다. 그래서 먼 과거까지 기울기가 덜 사라집니다.

---
## 8. GRU — 게이트 2개로 줄인 LSTM

GRU 는 셀 상태와 은닉 상태를 하나로 합치고, 게이트를 **리셋 $r_t$** 와 **업데이트 $z_t$** 2개로 줄였습니다. 임시 은닉 상태까지 **세 묶음**의 가중치를 쓰므로 RNN 의 3배입니다.

> 💡 **표기 차이** — 강의노트는 $h_t = (1 - z_t) \otimes h_{t-1} + z_t \otimes \tilde c_t$ 로 적고, PyTorch 문서는 $z_t$ 와 $1 - z_t$ 의 자리를 바꿔 적습니다 ($z_t$ 가 1 에 가까우면 이전 기억을 유지). 책마다 두 표기가 모두 쓰이며, 게이트가 반대 뜻으로 학습될 뿐 구조는 같습니다.

아래 두 클래스도 강의노트 그대로이고, `output_size` 만 바꿨습니다. LSTM 은 셀 상태의 초기값 `c0` 도 함께 넣습니다.

In [ ]:
class LSTMModel(nn.Module):                 # 강의노트의 LSTM 모델
    def __init__(self, input_size, hidden_size, num_layers, output_size):  # 크기 4가지
        super().__init__()                  # nn.Module 준비
        self.hidden_size = hidden_size      # 은닉 상태 칸 수
        self.num_layers = num_layers        # 층 수
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)  # 게이트 3개
        self.fc = nn.Linear(hidden_size, output_size)   # 은닉 상태 → 기온 1개

    def forward(self, x):                   # x 📐 [배치, 24, 변수 수]
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # h0 = 0
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # c0 = 0
        out, (hn, cn) = self.lstm(x, (h0, c0))   # (은닉, 셀) 상태를 묶어 넣고 받기
        return self.fc(out[:, -1, :])       # 마지막 시점 → 기온 1개

In [ ]:
class GRUModel(nn.Module):                  # 강의노트의 GRU 모델
    def __init__(self, input_size, hidden_size, num_layers, output_size):  # 크기 4가지
        super().__init__()                  # nn.Module 준비
        self.hidden_size = hidden_size      # 은닉 상태 칸 수
        self.num_layers = num_layers        # 층 수
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)  # 게이트 2개
        self.fc = nn.Linear(hidden_size, output_size)   # 은닉 상태 → 기온 1개

    def forward(self, x):                   # x 📐 [배치, 24, 변수 수]
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # h0 = 0
        out, hn = self.gru(x, h0)           # RNN 과 같은 모양 (셀 상태 없음)
        return self.fc(out[:, -1, :])       # 마지막 시점 → 기온 1개

---
## 9. 파라미터 수 — RNN : LSTM : GRU = 1 : 4 : 3

순환층 한 층의 파라미터 수 = **묶음 수 × 은닉 × (입력 + 은닉 + 2)**
- 은닉 × 입력 ($W_{xh}$) + 은닉 × 은닉 ($W_{hh}$) + 은닉 × 2 (PyTorch 는 편향을 `bias_ih` · `bias_hh` 두 개로 둡니다)
- 묶음 수: RNN **1** · LSTM **4** (게이트 3 + 임시 셀 상태) · GRU **3** (게이트 2 + 임시 은닉 상태)

5-1 의 작은 RNN (입력 1, 은닉 2) 이면 2 × (1 + 2 + 2) = **10개** 입니다. 은닉 32 로 세어 봅시다.

In [ ]:
def count_params(model):                    # 학습할 숫자(파라미터) 개수 세기
    return sum(p.numel() for p in model.parameters())  # 텐서마다 원소 수를 더함

specs = [("RNN", RNNModel, 1),              # (이름, 클래스, 묶음 수)
         ("LSTM", LSTMModel, 4),            # 게이트 3 + 임시 셀 상태
         ("GRU", GRUModel, 3)]              # 게이트 2 + 임시 은닉 상태
for name, Model, k in specs:                # 세 모델을 차례로
    m = Model(1, 32, 1, 1)                  # 입력 1 · 은닉 32 · 1층 · 출력 1
    total, fc = count_params(m), count_params(m.fc)  # 전체, 출력층 (32 + 1 = 33개)
    rec = total - fc                        # 순환층만
    print(f"{name:>4}: 전체 {total:,} = 순환층 {rec:,} + 출력층 {fc}")  # 나눠 보기
    check(f"{name} 순환층 = {k} × 32 × (1 + 32 + 2)", rec, k * 32 * (1 + 32 + 2), tol=0)  # ✅

### 9-1. 강의노트 설정으로 세어 보기
강의노트 실습은 입력 28 · 은닉 128 · **2층** · 출력 10 입니다. 2층째 순환층은 입력이 아래층의 은닉 128칸이므로 128 × (128 + 128 + 2) 개가 더해집니다.

In [ ]:
note = [count_params(M(28, 128, 2, 10)) for M in (RNNModel, LSTMModel, GRUModel)]  # 강의노트 설정
print("강의노트 설정 (RNN, LSTM, GRU):", [f"{v:,}" for v in note])  # 직접 센 개수
layer = 128 * (28 + 128 + 2) + 128 * (128 + 128 + 2)   # RNN 순환층 2층 (공식)
expect = [k * layer + 128 * 10 + 10 for k in (1, 4, 3)]  # 묶음 수 × 순환층 + 출력층
check("세 모델 파라미터 수 = 공식", note, expect, tol=0)  # ✅

🔍 **결과 읽는 법** — 어떤 설정이든 **순환층은 RNN : LSTM : GRU = 1 : 4 : 3** 입니다. 출력층(`fc`)은 세 모델이 똑같으므로, 전체 개수의 비율은 1 : 4 : 3 에 조금 못 미칩니다. 파라미터 수는 설정에 따라 달라지니, 이렇게 **직접 세어 보는** 습관을 들이세요.

---
## 10. 학습 준비 — 미니배치 · 비용함수 · 학습 루프  🎬 EP4

- `TensorDataset(X, y)` : 입력 창과 정답을 짝지은 데이터셋 · `DataLoader` : 64개씩 미니배치 (지난 실습과 같음)
- 미니배치를 섞어도 될까요? **창 안의 24시간 순서는 그대로**이고, 어떤 창들이 한 묶음이 될지만 섞입니다.
- 비용함수: 숫자 하나를 맞히는 **회귀**이므로 `nn.MSELoss()` (평균제곱오차)를 씁니다. 분류의 교차엔트로피 대신입니다.

In [ ]:
BATCH = 64                                  # 미니배치 크기
train_set = TensorDataset(X_train, y_train) # (입력 창, 정답) 짝들
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True,  # 64개씩, 창 순서 섞기
                          generator=torch.Generator().manual_seed(0))  # 섞는 순서 고정
X_batch, y_batch = next(iter(train_loader))  # 첫 미니배치 하나 꺼내 보기
print("미니배치:", tuple(X_batch.shape), "→", tuple(y_batch.shape))  # 📐 [64, 24, 1] → [64, 1]
print("한 에포크 =", len(train_loader), "번 업데이트")  # 학습 창 수 ÷ 64 (올림)
loss_fn = nn._______()                      # ✏️ 회귀용 비용함수: 평균제곱오차 (힌트: 평균제곱오차)

### 10-1. 학습 루프 4단계 — 지난 실습과 같은 뼈대

| 단계 | 코드 | 이론 |
|:-:|---|---|
| 1️⃣ 순전파 | `pred = model(X)` | 24시점을 차례로 → 마지막 은닉 상태 → 기온 |
| 2️⃣ 오차 계산 | `loss = loss_fn(pred, y)` | 평균제곱오차 |
| 3️⃣ 역전파 | `optimizer.zero_grad()` → `loss.backward()` | **BPTT**: 시간을 거슬러 24시점 모두의 기울기 |
| 4️⃣ 업데이트 | `optimizer.step()` | w ← w − α · (Adam이 다듬은 기울기) |

In [ ]:
def train_one_epoch(model, loader, optimizer):  # 학습 데이터 한 바퀴 = 1 에포크
    model.train()                           # 학습 모드
    total = 0.0                             # 오차 합계
    for X, y in loader:                     # 미니배치(64개)마다
        X, y = X.to(device), y.to(device)   # 계산 장치로 옮기기
        pred = model(X)                     # 1️⃣ 순전파: 1시간 뒤 기온 예측
        loss = loss_fn(pred, y)             # 2️⃣ 오차 (평균제곱오차)
        optimizer._________()               # ✏️ 3️⃣ 이전 기울기 지우기 (힌트: 기울기 0)
        loss.________()                     # ✏️ 3️⃣ 역전파 (BPTT)
        optimizer.____()                    # ✏️ 4️⃣ 가중치 업데이트 (힌트: 한 걸음)
        total += loss.item() * len(X)       # 평균 오차 × 개수 = 오차 합
    return total / len(loader.dataset)      # 이번 에포크의 평균 오차 (정규화 단위)

### 10-2. 채점 함수 — 평균 절대 오차를 °C 로
기준점과 **같은 잣대**로 채점합니다: |예측 − 정답| 의 평균 × 기온의 표준편차.

In [ ]:
def evaluate_mae(model, X, y):              # 평균 절대 오차 (°C)
    model.eval()                            # 평가 모드
    with torch.no_grad():                   # 채점만 (미분 추적 끄기)
        pred = model(X.to(device)).cpu()    # 창 전체를 한 번에 예측
    return ((pred - y).abs().mean() * ______).item()  # ✏️ × 표준편차 → °C (힌트: 기온의 표준편차)

### 10-3. 세 모델 학습 — 5 에포크씩 ⏸
강의노트처럼 세 모델을 **같은 조건**(은닉 32 · 1층 · Adam · 학습률 0.001 · 5 에포크)으로 차례로 학습합니다. 에포크마다 **검증** 오차를 보고, **시험** 오차는 맨 끝에 한 번만 잽니다.

⏸ 이 노트북을 만든 컴퓨터의 CPU 로 세 모델 합쳐 1분이 채 안 걸렸습니다. Colab CPU 에서는 몇 배 걸릴 수 있고, GPU 런타임이면 빠릅니다.

> ⚠️ 이 셀을 다시 실행하면 세 모델을 **처음부터** 다시 학습합니다 (셀 안에서 새로 만들기 때문).

In [ ]:
EPOCHS = 5                                  # 강의노트와 같은 5 에포크
results = {}                                # 모델별 결과를 모을 상자
for name, Model in [("RNN", RNNModel), ("LSTM", LSTMModel), ("GRU", GRUModel)]:  # 차례로
    torch.manual_seed(0)                    # 같은 출발점 (재현용)
    model = Model(1, 32, 1, 1).to(device)   # 입력 1 · 은닉 32 · 1층 · 출력 1
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # 강의노트와 같은 Adam · 0.001
    start = time.time()                     # ⏱ 시작
    for epoch in range(1, EPOCHS + 1):      # 에포크 1 ~ 5
        loss = train_one_epoch(model, train_loader, optimizer)  # 학습 한 바퀴
        val = evaluate_mae(model, X_val, y_val)  # 검증 채점 (°C)
        print(f"{name:>4} 에포크 {epoch}: 학습 오차 {loss:.4f} · 검증 오차 {val:.3f} °C")  # 진행
    sec = time.time() - start               # ⏱ 걸린 시간
    test = evaluate_mae(model, X_test, y_test)  # 시험 채점 — 맨 끝에 한 번만
    results[name] = dict(model=model, test=test, sec=sec, params=count_params(model))  # 기록

🔍 **결과 읽는 법** — 학습 오차는 정규화 단위의 평균제곱오차, 검증 오차는 °C 단위의 평균 절대 오차입니다. 첫 에포크에는 검증 오차가 기준점(검증 0.68 °C)과 비슷하다가, 둘째 에포크부터 0.5 °C 대로 확실히 낮아집니다. 매번 줄기만 하지는 않고 조금씩 오르내리기도 합니다 (RNN 은 4 → 5 에포크에 0.52 → 0.56 °C).

### 10-4. 결과 비교 — 기준점을 이겼을까?

In [ ]:
rows = [{"모델": "기준점 (지금 기온 그대로)", "시험 오차 (°C)": round(base_test, 2),
         "파라미터 수": 0, "학습 시간 (초)": 0}]     # 첫 줄: 기준점
for name, r in results.items():             # 세 모델을 한 줄씩
    rows.append({"모델": name, "시험 오차 (°C)": round(r["test"], 2),  # 시험 오차
                 "파라미터 수": r["params"], "학습 시간 (초)": round(r["sec"])})  # 크기 · 시간
for name, r in results.items():             # 자가검증: 셋 다 기준점보다 작게
    check_range(f"{name} 시험 오차 (°C)", r["test"], 0.40, 0.70)  # ✅ 기준점보다 확실히 작게
pd.DataFrame(rows)                          # 표로 보기

🔍 **결과 읽는 법** — 세 모델 모두 기준점(0.74 °C)보다 25 ~ 32 % 작은 **0.50 ~ 0.55 °C** 입니다. 이번 실행에서는 LSTM 이 0.50 °C 로 가장 좋았지만 차이는 0.05 °C 정도로 작습니다. 1시간 뒤를 맞히는 데는 최근 몇 시간이 거의 전부라 긴 기억이 크게 필요하지 않기 때문입니다. 대신 파라미터 수는 LSTM 이 RNN 의 약 4배, 학습 시간은 2배 넘게 걸렸습니다. (Colab · GPU 에서는 소수점 아래 숫자가 조금 다를 수 있습니다)

### 10-5. 시험 구간 첫 3일을 그래프로
실제 기온 · 기준점 · GRU 의 예측을 °C 로 되돌려 겹쳐 그립니다.

In [ ]:
k = slice(0, 72)                            # 시험 구간의 첫 72시간 (3일)
true = y_test[k, 0] * std[0] + mean[0]      # 실제 기온 (°C 로 되돌리기)
base = X_test[k, -1, 0] * std[0] + mean[0]  # 기준점: 지금 기온 그대로
gru = results["GRU"]["model"]               # 학습한 GRU
with torch.no_grad():                       # 예측만
    pred = gru(X_test[k].to(device)).cpu()[:, 0] * std[0] + mean[0]  # GRU 예측 (°C)
plt.figure(figsize=(11, 3.4))               # 가로로 긴 그래프
plt.plot(true, "k.-", label="actual"); plt.plot(base, "--", label="baseline (now)")  # 실제 · 기준점
plt.plot(pred, label="GRU")                 # 모델의 예측
plt.xlabel("hour"); plt.ylabel("T (°C)"); plt.legend()  # 축 이름 · 범례 (영문)
plt.show()                                  # 그리기

🔍 **결과 읽는 법** — 기준점(점선)은 실제 곡선을 **한 시간 늦게** 따라갑니다. 기온이 오르는 아침과 내리는 저녁에 특히 어긋나죠. GRU 는 지난 24시간의 흐름을 보고 오르내림을 미리 반영해, 대체로 실제 곡선에 더 가깝습니다 (가끔은 지나치게 앞서 가기도 합니다).

---
## 11. 실험실 — 더 어려운 문제 · 더 많은 정보 · 긴 기억  🎬 EP5

1~4장을 함수 두 개로 묶어, 설정만 바꿔 가며 실험합니다.
- `make_data(features, width, ahead)` : 열 고르기 → 시간 순서대로 나누기 · 정규화 → 창 만들기 (1~3장)
- `fit(Model, data, seed)` : 5 에포크 학습 → 시험 오차 °C (4장)

In [ ]:
def make_data(features, width=24, ahead=1):     # 1~3장을 한 번에 (⚠️ 기온을 맨 앞 열에)
    d = torch.tensor(df[features].values, dtype=torch.float32)  # 고른 열들  📐 [시간 수, 변수 수]
    m, s = d[:n_train].mean(dim=0), d[:n_train].std(dim=0)      # 학습 구간의 평균 · 표준편차
    zz = (d - m) / s                            # 정규화 (기온 열은 3장과 같은 값)
    parts = (zz[:n_train], zz[n_train:n_val], zz[n_val:])       # 학습 · 검증 · 시험 (시간 순서)
    return [make_windows(p, width, ahead) for p in parts]       # [(X, y), (X, y), (X, y)]

In [ ]:
def fit(Model, data, seed=0):                   # 4장의 학습을 한 번에 → 시험 오차 (°C)
    (X_tr, y_tr), _, (X_te, y_te) = data        # 학습 · (검증은 건너뜀) · 시험
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True,  # 미니배치
                        generator=torch.Generator().manual_seed(seed))  # 섞는 순서 고정
    torch.manual_seed(seed)                     # 같은 출발점
    model = Model(X_tr.shape[2], 32, 1, 1).to(device)  # 입력 수 = 변수 수 · 은닉 32 · 1층
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)   # 4장과 같은 설정
    for epoch in range(EPOCHS):                 # 5 에포크
        train_one_epoch(model, loader, optimizer)  # 학습 한 바퀴
    return evaluate_mae(model, X_te, y_te)      # 시험 오차 (°C)

### 11-1. 하루 뒤 기온 — 더 어려운 문제 ⏸
`ahead=24` 로 바꾸면 24시간 뒤를 맞힙니다. 기준점도 "하루 뒤 기온 = 지금 기온"이 됩니다.

> 💡 **새 파이썬 문법 — `f(*묶음)`**
>
> `day[2]` 는 시험용 `(X, y)` 묶음입니다. `persistence_mae(*day[2])` 처럼 앞에 `*` 를 붙이면 묶음을 풀어 `persistence_mae(X, y)` 로 넣습니다.

In [ ]:
t0 = time.time()                                # ⏱ 시작
day = make_data(["T (degC)"], ________)         # ✏️ 기온만 · 24시간 뒤 (힌트: 하루 = 24시간)
day_base = persistence_mae(*day[2])             # 기준점: 하루 뒤 = 지금 (시험 구간)
day_gru = fit(GRUModel, day)                    # GRU 학습 → 시험 오차
sec = time.time() - t0                          # ⏱ 걸린 시간
print(f"하루 뒤 — 기준점 {day_base:.2f} °C · GRU {day_gru:.2f} °C · ⏱ {sec:.0f}초")  # 비교
check_range("하루 뒤 기준점 (°C)", day_base, 2.0, 2.8)  # ✅ 1시간 뒤보다 훨씬 어려움

### 11-2. 기압 · 습도도 함께 — 정보를 더 주면?
`features` 에 기압 `p (mbar)` 와 습도 `rh (%)` 를 더합니다. 입력 모양이 `[N, 24, 3]` 이 되고, `fit` 이 `X.shape[2]` 로 `input_size` 를 3 에 맞춥니다.

In [ ]:
t0 = time.time()                                # ⏱ 시작
day3 = make_data(["T (degC)", "p (mbar)", "rh (%)"], ahead=24)  # 기온 + 기압 + 습도
print("입력 모양:", tuple(day3[0][0].shape))     # 📐 [N, 24, 3] — 변수 3개
day3_gru = fit(GRUModel, day3)                  # 같은 GRU · 같은 설정
sec = time.time() - t0                          # ⏱ 걸린 시간
print(f"변수 3개 — GRU {day3_gru:.2f} °C (기온만 {day_gru:.2f} °C) · ⏱ {sec:.0f}초")  # 하루 뒤
better = "좋아졌습니다" if day3_gru < day_gru else "이번에는 좋아지지 않았습니다"  # 실험 결과
print("🔸 기압 · 습도를 더하니", better)           # 오류가 아니라 실험 결과 안내

🔍 **결과 읽는 법** — 하루 뒤는 훨씬 어렵습니다. 기준점이 2.38 °C 이고, 기온만 본 GRU 는 2.34 °C 로 거의 차이가 없습니다. 기온 기록만으로는 내일의 날씨 변화를 알기 어렵죠. 기압 · 습도를 더하면 **2.16 °C** 로 기준점보다 0.2 °C 쯤 좋아집니다. 다른 측정값이 날씨 변화의 단서를 주기 때문으로 보입니다.

### 11-3. 긴 기억 시험 — 시간 순서를 뒤집으면? ⏸ (선택)
1시간 뒤를 맞히는 데 가장 중요한 정보는 **가장 최근 기온**입니다. 창의 순서를 **뒤집어** 넣으면 그 기온이 맨 앞에 오므로, 모델은 그것을 **23시간 동안 기억했다가** 마지막에 써야 합니다. 긴 기억 시험이죠.
- 『케라스 창시자에게 배우는 딥러닝』도 같은 데이터로 순서를 뒤집는 실험을 합니다.
- 결과가 씨앗(seed)마다 달라서 **씨앗 3개의 평균**으로 비교합니다. 9번 학습하므로 CPU 로 2분쯤 걸립니다.

In [ ]:
plain = [(X_train, y_train), (X_val, y_val), (X_test, y_test)]  # 4장과 같은 데이터
rev = [(X._______, y) for X, y in plain]        # ✏️ 창마다 순서 뒤집기 (힌트: 1번 축(시점))
print("원래 첫 창의 끝 3시간 :", X_train[0, -3:, 0])  # 가장 최근 기온이 맨 뒤
print("뒤집은 첫 창의 앞 3시간:", rev[0][0][0, :3, 0])  # 가장 최근 기온이 맨 앞
t0, avg = time.time(), {}                       # ⏱ 시작 · 모델별 평균 시험 오차
for name, Model in [("RNN", RNNModel), ("LSTM", LSTMModel), ("GRU", GRUModel)]:  # 세 모델
    maes = [fit(Model, rev, seed) for seed in (0, 1, 2)]  # 씨앗 3개로 3번 학습
    avg[name] = sum(maes) / len(maes)           # 평균
    print(f"{name:>4}: 씨앗별 {[round(v, 2) for v in maes]} → 평균 {avg[name]:.2f} °C"  # 뒤집으면
          f" (순서대로 {results[name]['test']:.2f} °C)")  # 4장 결과와 비교
worst = max(avg, key=avg.get)                   # 평균 오차가 가장 큰 모델
print(f"⏱ {time.time() - t0:.0f}초 · 🔸 가장 크게 무너진 모델: {worst}")  # 실험 결과
check_range("뒤집은 RNN 의 평균 오차 (°C)", avg["RNN"], 0.6, 3.0)  # ✅ 순서대로(약 0.5)보다 나쁨

🔍 **결과 읽는 법** — 순서대로 넣으면 세 모델 모두 0.50 ~ 0.55 °C 였죠. 순서를 뒤집으면 **RNN 은 평균 1.12 °C** 로 기준점(0.74 °C)보다도 나빠집니다. 맨 앞의 최근 기온을 은닉 상태 하나로 23시간 동안 끌고 오다가 흐려진 것입니다 (기울기 소실 · 장기 의존성 문제). 게이트가 있는 **LSTM(0.69 °C) 과 GRU(0.63 °C)** 는 훨씬 덜 무너집니다.
그런데 씨앗별로 보면 RNN 이 0.69 °C 로 버틴 경우도, GRU 가 0.77 °C 로 흔들린 경우도 있습니다. 한 번의 결과로 결론 내리지 말고 **여러 번 재 보는 습관**이 중요합니다. (Colab 에서는 씨앗별 숫자가 꽤 다르게 나올 수 있습니다)

### 11-4. 🏆 도전 과제
1. **창 길이** — `make_data(FEATURES, width=72)` (3일)이나 `width=168` (7일)로 1시간 뒤를 맞혀 보세요. 오차와 학습 시간은 어떻게 바뀔까요? (`fit` 은 그대로 씁니다)
2. **더 큰 모델** — 강의노트 과제처럼 `fit` 안의 `Model(X_tr.shape[2], 32, 1, 1)` 에서 은닉 32 → 64 · 128, 층 1 → 2 로 바꾸면? 파라미터 수는 9장의 공식으로 먼저 계산해 보세요.
3. **GRU 손 계산** — PyTorch 문서의 GRU 식으로 7-1 처럼 손 계산해 `nn.GRU` 와 비교해 보세요. 가중치는 (리셋 $r$, 업데이트 $z$, 임시 $n$) 순서로 3묶음이고, $n_t = \tanh(W_{in}x_t + b_{in} + r_t \otimes (W_{hn}h_{t-1} + b_{hn}))$, $h_t = (1 - z_t) \otimes n_t + z_t \otimes h_{t-1}$ 입니다.

---
## 12. 핵심 요약

### 이론 ↔ 코드 대응표 (RNN 판)
| 이론 (강의노트) | PyTorch 코드 |
|---|---|
| 시계열 · many-to-one | `make_windows(series, width=24, ahead=1)` → 입력 `[N, 24, 변수 수]` · 정답 `[N, 1]` |
| 시간 순서대로 나누기 · 정규화 | 앞 70 % · 20 % · 10 % · 평균 · 표준편차는 학습 구간에서만 |
| 은닉 상태 $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b)$ | `nn.RNN(1, 32, batch_first=True)` · `weight_ih_l0` · `weight_hh_l0` · `bias_ih_l0 + bias_hh_l0` |
| 파라미터 공유 | 모든 시점에 같은 가중치 (`for t in range(...)`) |
| 마지막 시점의 은닉 상태 → 예측 | `out[:, -1, :]` → `nn.Linear(32, 1)` |
| LSTM: 셀 상태 · 게이트 3개 | `nn.LSTM` · `out, (hn, cn)` · `c = f * c + i * g` · `h = o * tanh(c)` |
| GRU: 게이트 2개 | `nn.GRU` · `out, hn` |
| 파라미터 수 | 묶음 수 × 은닉 × (입력 + 은닉 + 2) → RNN : LSTM : GRU = 1 : 4 : 3 |
| BPTT (시간을 거슬러 역전파) | `loss.backward()` |
| 회귀 비용함수 · 채점 | `nn.MSELoss()` · 평균 절대 오차 × 표준편차 = °C |
| 상식 수준의 기준점 | `X[:, -1, :1]` (지금 기온 그대로) |

### 학습 루프 4단계 (지난 실습과 같은 뼈대!)
```python
pred = model(X)            # 1️⃣ 순전파
loss = loss_fn(pred, y)    # 2️⃣ 오차 계산 (MSELoss)
optimizer.zero_grad()      # 3️⃣ 역전파: 기울기 지우고
loss.backward()            #           기울기 계산 (BPTT)
optimizer.step()           # 4️⃣ 가중치 업데이트
```

### ⚠️ 자주 하는 실수 5가지
1. **무작위로 섞어 나누기** → 시험 기간 바로 앞뒤가 학습에 섞여 점수가 부풀려집니다. 시계열은 **시간 순서대로** 나누기!
2. **전체 데이터로 정규화** → 시험 구간의 평균 · 표준편차를 미리 본 셈입니다. 평균 · 표준편차는 **학습 구간에서만**
3. `batch_first=True` 를 빠뜨림 → `[배치, 시점, 변수]` 를 `[시점, 배치, 변수]` 로 읽습니다. `h0` 를 직접 넣는 강의노트 클래스는 `Expected hidden size …` 오류가 나고, `h0` 를 생략하면 **오류 없이** 엉뚱하게 학습합니다.
4. 변수 축을 빼고 `[N, 24]` 로 넣음 → `input.size(-1) must be equal to input_size` 오류. 변수가 1개여도 `[N, 24, 1]`
5. 정규화된 오차를 °C 로 착각 → **× 표준편차** 해야 °C 입니다. 그리고 모델 점수는 늘 **기준점과 비교**하기

### 강의노트의 실습 코드와 무엇이 다른가요?
- 데이터: 손글씨 숫자 그림(행 28개를 시점으로) → **예나 기온 기록** (진짜 시계열)
- `input_size` 28 → **1** (5장 실험은 3) · 출력 `num_classes` 10 → **`output_size` 1** · `CrossEntropyLoss` → **`MSELoss`** · 정확도 → **평균 절대 오차 (°C)**
- 은닉 128 · 2층 → **은닉 32 · 1층** (기온 1개를 넣는 작은 문제라 충분하고, CPU 에서도 빠름) · 미니배치 100 → 64
- 모델 클래스 구조 · Adam (학습률 0.001) · 5 에포크는 **그대로**
- 손 계산으로 `nn.RNN` · `nn.LSTM` 확인 · 기준점 비교 · ✅ 자가검증을 더했습니다.

### 다음 실습 예고
오늘은 24시간을 보고 **한 시점**을 맞혔습니다 (many-to-one). 다음 실습에서는 여러 시점을 **한꺼번에** 예측하는 **many-to-many** 구조를 다룹니다.